# Sheet 1.4: A neural network from scratch (NumPy only)

**Roadmap topic 1.4** (Fanout module 05). Watch first: Karpathy's *makemore* parts 1-3 (part 3 is about activations, gradients and init). Read: Nielsen ch. 1-3, the RMSNorm paper, Kaiming init (He et al. 2015, sec. 2.2).

This sheet **is your Step 1 portfolio artifact**. When everything passes, move the code into a public repo (`mlp-from-scratch`) with tests, a README and the ablation plots from Exercise 5.

**Rules:** your library code uses **NumPy only**. PyTorch appears only inside `CHECK` cells as a reference. Fill every `# YOUR CODE` cell, never edit `CHECK` cells.

In [ ]:
import math, numpy as np, torch
rng = np.random.default_rng(0)
torch.set_default_dtype(torch.float64)

## Exercise 1: Layers with forward and backward

Each layer has `forward(x)` (caching what backward needs), `backward(dout)` returning `dx` and filling `self.grads` (a dict matching `self.params`). Implement:
- `Linear(n_in, n_out)`: `y = x W + b`, Kaiming-uniform init for ReLU-family nets.
- `ReLU`, `GELU` (tanh approximation, as `torch.nn.functional.gelu(approximate="tanh")`).
- `RMSNorm(d, eps=1e-6)`: `y = x / sqrt(mean(x²) + eps) * g`, learnable `g` initialised to ones.
- `SoftmaxCE`: `forward(logits, y)` returns the mean loss; `backward()` returns `dlogits`.

In [ ]:
class Layer:
    params, grads = {}, {}
class Linear(Layer):
    def __init__(self, n_in, n_out, rng=rng):
        # YOUR CODE
        raise NotImplementedError
class ReLU(Layer):
    pass  # YOUR CODE
class GELU(Layer):
    pass  # YOUR CODE
class RMSNorm(Layer):
    def __init__(self, d, eps=1e-6):
        # YOUR CODE
        raise NotImplementedError
class SoftmaxCE:
    pass  # YOUR CODE

In [ ]:
# ---- CHECK 1 (do not edit) ----
def _check1():
    F = torch.nn.functional
    x = rng.normal(size=(16, 12)) * 2; d = rng.normal(size=(16, 12))
    for name, layer, ref in (("ReLU", ReLU(), torch.relu), ("GELU", GELU(), lambda t: F.gelu(t, approximate="tanh"))):
        xt = torch.tensor(x, requires_grad=True); ref(xt).backward(torch.tensor(d))
        assert np.allclose(layer.forward(x), ref(torch.tensor(x)).numpy()) and np.allclose(layer.backward(d), xt.grad.numpy()), name
    lin = Linear(12, 7); xt = torch.tensor(x, requires_grad=True)
    W = torch.tensor(lin.params["W"], requires_grad=True); b = torch.tensor(lin.params["b"], requires_grad=True)
    dd = rng.normal(size=(16, 7)); (xt @ W + b).backward(torch.tensor(dd))
    assert np.allclose(lin.forward(x), (torch.tensor(x) @ W + b).detach().numpy())
    assert np.allclose(lin.backward(dd), xt.grad.numpy()) and np.allclose(lin.grads["W"], W.grad.numpy()) and np.allclose(lin.grads["b"], b.grad.numpy()), "Linear grads"
    assert abs(Linear(1000, 1000).params["W"].std() - math.sqrt(2 / 1000)) < 2e-3, "use Kaiming init (std = sqrt(2/fan_in))"
    rn = RMSNorm(12); rn.params["g"] = rng.normal(size=12)
    xt = torch.tensor(x, requires_grad=True); g = torch.tensor(rn.params["g"], requires_grad=True)
    out = F.rms_norm(xt, (12,), g, 1e-6); out.backward(torch.tensor(d))
    assert np.allclose(rn.forward(x), out.detach().numpy()), "RMSNorm forward"
    assert np.allclose(rn.backward(d), xt.grad.numpy()) and np.allclose(rn.grads["g"], g.grad.numpy()), "RMSNorm backward"
    z = rng.normal(size=(16, 5)) * 30; y = rng.integers(0, 5, 16); ce = SoftmaxCE()
    zt = torch.tensor(z, requires_grad=True); lt = F.cross_entropy(zt, torch.tensor(y)); lt.backward()
    assert abs(ce.forward(z, y) - lt.item()) < 1e-10 and np.allclose(ce.backward(), zt.grad.numpy()), "SoftmaxCE"
    print("PASS")
_check1()

## Exercise 2: AdamW and a warmup + cosine schedule

- `lr_at(step, max_lr, warmup, total, min_lr)`: linear warmup from 0 to `max_lr` over `warmup` steps (step 0 gives `max_lr/warmup`), then cosine decay to `min_lr` at `total`, and `min_lr` after.
- `AdamW(layers, lr, betas, eps, weight_decay)` with `step(lr)` updating every array in every layer's `params` from `grads`. Apply weight decay **only to 2-D weights** (not biases, not norm gains), as every LLM codebase does.

In [ ]:
def lr_at(step, max_lr, warmup, total, min_lr):
    # YOUR CODE
    raise NotImplementedError
class AdamW:
    def __init__(self, layers, betas=(0.9, 0.999), eps=1e-8, weight_decay=0.01):
        # YOUR CODE
        raise NotImplementedError
    def step(self, lr):
        # YOUR CODE
        raise NotImplementedError

In [ ]:
# ---- CHECK 2 (do not edit) ----
def _check2():
    s = [lr_at(i, 1e-3, 10, 100, 1e-4) for i in range(120)]
    assert abs(s[0] - 1e-4) < 1e-12 and abs(s[9] - 1e-3) < 1e-12, "warmup wrong"
    assert abs(s[55] - (1e-4 + 0.5 * 9e-4 * (1 + math.cos(math.pi * 0.5)))) < 1e-12 and s[100] == 1e-4 and s[119] == 1e-4, "cosine wrong"
    assert all(a >= b for a, b in zip(s[9:], s[10:])), "schedule must not increase after warmup"
    L1, N, L2 = Linear(6, 8), RMSNorm(8), Linear(8, 3)
    tp = {(i, k): torch.tensor(v.copy(), requires_grad=True) for i, L in enumerate((L1, N, L2)) for k, v in L.params.items()}
    decay = [p for (i, k), p in tp.items() if p.dim() == 2]; nodecay = [p for (i, k), p in tp.items() if p.dim() != 2]
    topt = torch.optim.AdamW([{"params": decay, "weight_decay": 0.1}, {"params": nodecay, "weight_decay": 0.0}], lr=1e-2)
    opt = AdamW([L1, N, L2], weight_decay=0.1)
    for step in range(50):
        G = {k: rng.normal(size=v.shape) for k, v in tp.items()}
        for (i, k), g in G.items():
            (L1, N, L2)[i].grads[k] = g; tp[(i, k)].grad = torch.tensor(g)
        lr = lr_at(step, 1e-2, 5, 50, 1e-3)
        for gr in topt.param_groups: gr["lr"] = lr
        topt.step(); opt.step(lr)
    for (i, k), p in tp.items():
        assert np.allclose((L1, N, L2)[i].params[k], p.detach().numpy(), atol=1e-10), f"layer {i} param {k} differs from torch AdamW"
    print("PASS")
_check2()

## Exercise 3: Initialisation and activation statistics

`activation_stds(init, depth, width, act, norm)` pushes a batch of 512 standard-normal inputs through `depth` layers of `Linear(width, width)` + activation (+ `RMSNorm` before the activation if `norm=True`) and returns the list of activation stds per layer. `init` is one of `"normal"` (N(0,1)), `"xavier"` (std `sqrt(1/fan_in)`), `"kaiming"` (std `sqrt(2/fan_in)`); override the Linear weights after construction.

In [ ]:
def activation_stds(init, depth=20, width=256, act="relu", norm=False, seed=0):
    # YOUR CODE
    raise NotImplementedError

In [ ]:
# ---- CHECK 3 (do not edit) ----
def _check3():
    k = activation_stds("kaiming"); xv = activation_stds("xavier"); nm = activation_stds("normal")
    assert 0.3 < k[-1] / k[0] < 3, f"Kaiming+ReLU should keep scale: {k[0]:.3f} -> {k[-1]:.3f}"
    assert xv[-1] / xv[0] < 0.01, "Xavier+ReLU should shrink by about 2x per layer... check your init"
    assert nm[-1] > 1e10, "N(0,1) init should explode"
    nn_ = activation_stds("normal", norm=True)
    assert 0.3 < nn_[-1] < 3, "RMSNorm should rescue the bad init"
    print(f"PASS  layer-20 std: kaiming {k[-1]:.2f}, xavier {xv[-1]:.1e}, normal {nm[-1]:.1e}, normal+RMSNorm {nn_[-1]:.2f}")
_check3()

**Written (3a):** Derive why ReLU needs `2/fan_in` and not `1/fan_in`. Then explain why pre-norm transformers are much less sensitive to init than this plain MLP.

**Your answer:**

## Exercise 4: Put it together and train

`MLP(sizes, act, norm)` stacks your layers (RMSNorm before each activation if `norm`), with `forward`, `backward`, and `layers` (a flat list for the optimizer). `fit(model, X, y, steps, batch, max_lr, ...)` trains with minibatches, AdamW and your schedule, returning the loss history.

The check trains on a 3-class spiral (non-linearly separable) and needs ≥ 97% held-out accuracy.

In [ ]:
def spiral(n=600, seed=0):
    r = np.random.default_rng(seed); X, Y = [], []
    for c in range(3):
        t = np.linspace(0, 1, n); a = t * 4 + c * 2 * math.pi / 3 + r.normal(0, .15, n)
        X.append(np.c_[t * np.sin(a), t * np.cos(a)]); Y.append(np.full(n, c))
    X, Y = np.concatenate(X), np.concatenate(Y); p = r.permutation(len(Y)); return X[p], Y[p]

In [ ]:
class MLP:
    def __init__(self, sizes, act="gelu", norm=True, rng=rng):
        # YOUR CODE
        raise NotImplementedError
def fit(model, X, y, steps=3000, batch=128, max_lr=3e-3, warmup=100, wd=1e-4, seed=0):
    # YOUR CODE: return list of losses
    raise NotImplementedError
def predict(model, X):
    # YOUR CODE
    raise NotImplementedError

In [ ]:
# ---- CHECK 4 (do not edit) ----
def _check4():
    X, Y = spiral(); Xtr, Ytr, Xte, Yte = X[:1400], Y[:1400], X[1400:], Y[1400:]
    m = MLP([2, 128, 128, 3]); hist = fit(m, Xtr, Ytr)
    acc = (predict(m, Xte) == Yte).mean()
    assert np.isfinite(hist).all() and np.mean(hist[-100:]) < np.mean(hist[:100]), "loss did not decrease"
    assert acc >= 0.97, f"held-out accuracy {acc:.3f} < 0.97"
    print(f"PASS  held-out accuracy {acc:.3f}")
_check4()

## Exercise 5 (Colab or laptop): MNIST, Fashion-MNIST and ablations

1. Load MNIST and Fashion-MNIST (`torchvision.datasets` or `tf.keras.datasets`, then convert to NumPy). Train `MLP([784, 512, 512, 10])` with your library. **Targets: ≥ 98.0% on MNIST, ≥ 89.0% on Fashion-MNIST test.**
2. Ablate one component at a time, 3 seeds each: no RMSNorm, no warmup, no bias correction in AdamW, no weight decay, ReLU vs GELU. One plot per ablation (val loss vs step, mean ± std) and a summary table.
3. Create the public repo: library code in `mlp/`, tests in `tests/` (copy the checks above as pytest functions, plus an "overfit one batch to zero loss" test), a README with the results table and plots, and a one-command reproduce script.

In [ ]:
# Colab: MNIST / Fashion-MNIST training and ablations with YOUR NumPy library

## Step 1 exit test

Open `step1_diagnostic.ipynb` (in your project files, next to this sheet) and pass all 8 tasks **without help**. Any FAIL sends you back to the matching topic:
- task 1 → sheets 0.3 and 0.4 · task 2 → 0.3 · task 3 → 1.1 · task 4 → 1.3 · task 5 → 0.5 and this sheet · task 6 → 0.2 · task 7 → 1.2 · task 8 → 1.3 and 0.5.

---
### Hints for self-grading
- 3a: For `y = Wx`, `Var(y) = fan_in · Var(W) · E[x²]`. If `x = relu(z)` with symmetric `z`, then `E[x²] = ½ Var(z)`, so `Var(y) = ½ · fan_in · Var(W) · Var(z)`. Keeping it constant needs `Var(W) = 2/fan_in`. In pre-norm transformers each block's input is normalised and the residual stream adds rather than multiplies, so bad scales do not compound across depth.